# 数据清洗与预处理


在经过了数据探索以后，我们可以准备数据清洗和预处理。
首先，我们需要复制一个表格，要保留原数据不被改动。


In [1]:
from pathlib import Path

CURRENT_DIR = Path().cwd()
if CURRENT_DIR.name == "notebooks":
    #当前在notebooks/下面
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    #当前在根目录
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

In [2]:
import pandas as pd
df_train = pd.read_csv(RAW_DIR / "train.csv")

## 列名错误处理
我发现有的列名不统一，至少目前我发现的有，在表中写的是KitchenAbvGr，在data_desctiption.txt中写的是Kitchen，而表中写的BedroomAbvGr，在数据描述中写的是Bedroom.我们把他统一成数据描述中的样子。

In [3]:
df_train.rename(columns={
    "BedroomAbvGr": "Bedroom",
    "KitchenAbvGr": "Kitchen"
}, inplace=True)

## 缺失值处理
首先，我们填补一下缺失值。这个缺失值里面，大部分NaN都是有具体含义的，表示概念或者属性不适用。我们先把所有含有缺失值的列缺失值统计出来：

In [4]:
print("Columns with NA values:")
series = df_train.isna().mean()
series = series[series != 0]
print(series.sort_values().to_string())

Columns with NA values:
Electrical      0.000685
MasVnrArea      0.005479
BsmtCond        0.025342
BsmtFinType1    0.025342
BsmtQual        0.025342
BsmtExposure    0.026027
BsmtFinType2    0.026027
GarageType      0.055479
GarageQual      0.055479
GarageFinish    0.055479
GarageCond      0.055479
GarageYrBlt     0.055479
LotFrontage     0.177397
FireplaceQu     0.472603
MasVnrType      0.597260
Fence           0.807534
Alley           0.937671
MiscFeature     0.963014
PoolQC          0.995205


我们挨个处理。

* MiscFeature表示其他地方没有涉及的杂项，这个大部分数据都没有，也很合理。这个类的NA表示就是None，就是没有，因为该有的信息其他部分已经讲完了。

    这个我们把NA改名记作"NoMisc"，其他的保留，就可以消除缺失值了

* MasVnrType是指墙外面贴的装饰性石材的类型，如果没有就是None，很多房子都没有，这是所以是0.59

    这个有些特殊，专门说了None，NA和None共存，因此不能把NA改名。
* MasVnrArea是墙外石材的面积，这个缺失值非常少，要首先看看MasVnrType是不是None，不是None还缺失的才是真缺失。

* Alley是指房子挨着的小巷，绝大部分房子都没有这东西。

    大部分记成NoAlley,其他保留不变

* PoolQC指游泳池评级，绝大多数房子没有配套的。

    数据描述是：
    * Ex	Excellent
    * Gd	Good
    * TA	Average/Typical
    * Fa	Fair
    * NA	No Pool

    我本来决定把这些值量化，对应Ex = 4, Gd = 3, TA = 2, Fa = 1, NA = 0
    因为他们本身是有顺序的分类变量，这样处理是可以的。但是后来又想到，这样相当于假设不同类别的举例是相等的，不管这个合理与否，都应该是特征工程和模型阶段的事情，所以我们这里还是只把NA换成NoPool

* FireplaceQu缺的也比较多，这个是NA指的是没有对应设施，0.45的房子没有这个设施。

    理由相同，我们把NA换成NoFireplace，不做等距假设。

* GarageType以及Garage系列，这个系列里面NA也是表示没有车库，不是真正的缺失值。大概5.5%。

    由于具有明确的意义，我们不能删去，也许可以换个名称为"NoGarage"。包括其他Garage系列，GarageQual, GarageYrBlt, GarageFinish, GarageType, GarageCond，处理方法相同

* Bsmt系列，包括 BsmtFinType1, BsmtQual, BsmtCond和 BsmtFinType2, BsmtExposure，其中前3个是0.025342，后2个是0.026027。经过查阅发现全都是没有地下室。

    处理方法相同，全部填充为"NoBsmt"

* LotFrontage指的是房子到街道的距离，这个完全没说是什么类型数据，也没解释NA的含义，看来这个NA是真的彻底的数据缺失。

  这个数值缺失不多，并且意义明确，甚至搞不好可能服从正态分布，我们采用中位数补充上

* Electrical是指供电类型，这个没有说明NA是什么意思，空缺的值应该是真的缺失值了。

    极少数缺失值，且不好定义NA的含义，是纯缺失，我们决定删除为0的这些行
接下来我们开始执行

In [5]:
# 处理MiscFeature
# 取出对应列，对该列赋值, 使用fillna()
df_train["MiscFeature"] =  df_train["MiscFeature"].fillna("NoMisc")

# 处理Alley
df_train["Alley"] = df_train["Alley"].fillna("NoAlley")

# PoolQC把NA填成NoPool
df_train["PoolQC"] = df_train["PoolQC"].fillna("NoPool")

#FireplaceQu把NA填成NoFireplace
df_train["FireplaceQu"] = df_train["FireplaceQu"].fillna("NoFireplace")

# 处理Garage系列: GarageType, GarageQual, GarageYrBlt, GarageFinish, GarageCond
df_train["GarageType"] = df_train["GarageType"].fillna("NoGarage")
df_train["GarageQual"] = df_train["GarageQual"].fillna("NoGarage")
df_train["GarageYrBlt"] = df_train["GarageYrBlt"].fillna("NoGarage")
df_train["GarageFinish"] = df_train["GarageFinish"].fillna("NoGarage")
df_train["GarageCond"] = df_train["GarageCond"].fillna("NoGarage")

# 处理Bsmt系列：BsmtFinType1, BsmtQual, BsmtCond和 BsmtFinType2, BsmtExposure
df_train["BsmtFinType1"] = df_train["BsmtFinType1"].fillna("NoBsmt")
df_train["BsmtQual"] = df_train["BsmtQual"].fillna("NoBsmt")
df_train["BsmtCond"] = df_train["BsmtCond"].fillna("NoBsmt")
df_train["BsmtFinType2"] = df_train["BsmtFinType2"].fillna("NoBsmt")
df_train["BsmtExposure"] = df_train["BsmtExposure"].fillna("NoBsmt")

# 统计
print("Columns with NA values:")
series = df_train.isna().mean()
series = series[series != 0]
print(series.sort_values().to_string())

Columns with NA values:
Electrical     0.000685
MasVnrArea     0.005479
LotFrontage    0.177397
MasVnrType     0.597260
Fence          0.807534


现在还剩下
* Electrical       0.000685
* MasVnrArea       0.005479
* LotFrontage      0.177397
* FireplaceQu      0.472603
* MasVnrType     0.597260
* Fence            0.807534
* PoolQC           0.995205

In [6]:
# 删去Electrical 列中，为空的行
df_train =  df_train[~df_train["Electrical"].isna()]

# 取出LotFrontage列的中位数
LotFrontage_median = df_train["LotFrontage"].median()
# 然后把所有为NaN的替换成中位数
df_train["LotFrontage"] = df_train["LotFrontage"].fillna(LotFrontage_median)

# Fence，NA fillna为NoFence
df_train["Fence"] = df_train["Fence"].fillna("NoFence")

# 统计
print("Columns with NA values:")
series = df_train.isna().mean()
series = series[series != 0]
print(series.sort_values().to_string())

Columns with NA values:
MasVnrArea    0.005483
MasVnrType    0.596984


这个MasVnrArea和MasVnrType比较特殊，因为MasVnrType里面既有None，也有NA，经过查阅元数据，发现NA非常少，大部分都是None，而似乎pandas把两种都合并在一起了，但只有NA才是真缺失，None只是NoMasVnr。既然pandas已经混淆了这两列，我们干脆先把所有的MasVnrType空缺值都充填为NoMasVnr，然后再用MasVnrArea删除行。因为探究发现，基本MasVnrArea的NA都来自于MasVnrType的空缺。

In [7]:
# 把所有pandas认为是空缺的填补上NoMasVnr
df_train["MasVnrType"] = df_train["MasVnrType"].fillna("NoMasVnr")

# MasVnrArea直接删除空缺行
df_train = df_train[~df_train["MasVnrArea"].isna()]
# 统计
print("Columns with NA values:")
series = df_train.isna().mean()
series = series[series != 0]
print(series.sort_values().to_string())

Columns with NA values:
Series([], )


In [8]:
df_train.shape

(1451, 81)

好，这样我们就处理完了所有的空缺值。我们总共删了9行数据，把一些类别列的NA换成了字符串，把一些数值列的NA填补了一下。还有一些类别列的类别，我们顺带换成了数值，因为那些类别本身就有一定顺序和含义。

本来接下来应该处理类别变量了，不过我在处理的过程中，发现这里面不仅有str和纯数值，还有一个object，我们首先需要把他变成数值；

In [9]:
pd.set_option("display.max_rows", None)
df_train.dtypes

Id                 int64
MSSubClass         int64
MSZoning             str
LotFrontage      float64
LotArea            int64
Street               str
Alley                str
LotShape             str
LandContour          str
Utilities            str
LotConfig            str
LandSlope            str
Neighborhood         str
Condition1           str
Condition2           str
BldgType             str
HouseStyle           str
OverallQual        int64
OverallCond        int64
YearBuilt          int64
YearRemodAdd       int64
RoofStyle            str
RoofMatl             str
Exterior1st          str
Exterior2nd          str
MasVnrType           str
MasVnrArea       float64
ExterQual            str
ExterCond            str
Foundation           str
BsmtQual             str
BsmtCond             str
BsmtExposure         str
BsmtFinType1         str
BsmtFinSF1         int64
BsmtFinType2         str
BsmtFinSF2         int64
BsmtUnfSF          int64
TotalBsmtSF        int64
Heating              str


这个GarageYrBlt是object类型，因为我们之前fillna()的时候，把Garage系列的空白全都填上"NoGarage"了。我的解决方案是，再添加一个列变量`HasGarage`，有就是1，没有就是0.

In [10]:
#生成一个布尔值，再根据此添加新列
df_train["HasGarage"] = 1 * (df_train["GarageYrBlt"] != "NoGarage")
df_train["HasGarage"].dtypes

dtype('int64')

然后还要把原来的里的GarageYrBlt换回去，我们就换成数字0吧

In [11]:
df_train["GarageYrBlt"] = df_train["GarageYrBlt"].replace("NoGarage", 0)
df_train["GarageYrBlt"] = df_train["GarageYrBlt"].astype(int)
df_train.dtypes

Id                 int64
MSSubClass         int64
MSZoning             str
LotFrontage      float64
LotArea            int64
Street               str
Alley                str
LotShape             str
LandContour          str
Utilities            str
LotConfig            str
LandSlope            str
Neighborhood         str
Condition1           str
Condition2           str
BldgType             str
HouseStyle           str
OverallQual        int64
OverallCond        int64
YearBuilt          int64
YearRemodAdd       int64
RoofStyle            str
RoofMatl             str
Exterior1st          str
Exterior2nd          str
MasVnrType           str
MasVnrArea       float64
ExterQual            str
ExterCond            str
Foundation           str
BsmtQual             str
BsmtCond             str
BsmtExposure         str
BsmtFinType1         str
BsmtFinSF1         int64
BsmtFinType2         str
BsmtFinSF2         int64
BsmtUnfSF          int64
TotalBsmtSF        int64
Heating              str


这样就可以了，可以回去专心处理变量类别的问题了

## 类别变量处理
接下来，我们来专门处理一下类别的问题，有的类别变量适合当做离散数值变量，有的则数值变量适合当做类别变量处理，我们前面处理缺失值的时候已经处理了一部分，现在集中处理完剩下的部分。

首先，我们先取出所有的类别变量

In [12]:
df_categorical = df_train.select_dtypes("str")
df_categorical.shape

(1451, 43)

In [13]:
pd.set_option("display.max_columns", None)
df_categorical.describe()

,MSZoning,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinType2,Heating,HeatingQC,CentralAir,Electrical,KitchenQual,Functional,FireplaceQu,GarageType,GarageFinish,GarageQual,GarageCond,PavedDrive,PoolQC,Fence,MiscFeature,SaleType,SaleCondition
count,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451,1451
unique,5,2,3,4,4,2,5,3,25,9,8,5,8,6,8,15,16,4,4,5,6,5,5,5,7,7,6,5,2,5,4,7,6,7,4,6,6,3,4,5,5,9,6
top,RL,Pave,NoAlley,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Fam,1Story,Gable,CompShg,VinylSd,VinylSd,NoMasVnr,TA,TA,PConc,TA,TA,No,Unf,Unf,GasA,Ex,Y,SBrkr,TA,Typ,NoFireplace,Attchd,Unf,TA,TA,Y,NoPool,NoFence,NoMisc,WD,Normal
freq,1145,1445,1361,918,1302,1450,1045,1373,225,1251,1436,1212,721,1133,1425,509,498,863,905,1273,638,648,1303,945,427,1247,1419,734,1356,1326,734,1352,685,863,605,1302,1317,1331,1444,1170,1397,1261,1193


经过我的逐个排查，我发现这些变量的类别可能具有大小关系：
* LotShape
* Utilities
* LandSlope
* ExterQual
* ExterCond
* BsmtQual
* BsmtCond
* BsmtExposure
* BsmtFinType1
* BsmtFinType2
* HeatingQC
* KitchenQual
* Functional
* FireplaceQu
* GarageFinish
* GarageQual
* GarageCond
* PavedDrive
* PoolQC
* Fence

进一步再看看数值变量，有没有实际上应该作为类别变量的：

In [14]:
df_train.select_dtypes("number").describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,Bedroom,Kitchen,TotRmsAbvGrd,Fireplaces,GarageYrBlt,GarageCars,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice,HasGarage
count,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.00000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000
mean,729.374225,56.933150,69.845624,10507.808408,6.093728,5.579600,1971.092350,1984.760165,103.75672,442.274983,46.838043,567.197105,1056.310131,1161.551344,346.742247,5.880772,1514.174363,0.423846,0.057891,1.562371,0.381116,2.866988,1.046175,6.516885,0.612681,1867.914542,1.764990,472.525155,94.412819,46.422467,21.952447,3.430737,15.154376,2.776017,43.758787,6.319090,2007.815300,180624.101999,0.944176
std,421.737858,42.350366,22.044429,9992.987081,1.381467,1.113415,30.190266,20.651334,181.10815,455.369276,161.777324,442.209192,437.914609,385.002532,436.453277,48.771560,525.799521,0.518842,0.239449,0.550301,0.502574,0.815086,0.219571,1.626576,0.643443,454.981043,0.748728,214.171752,125.436852,66.060150,61.108223,29.406940,55.917522,40.301212,497.648467,2.700353,1.329376,79312.128273,0.229660
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.00000,0.000000,0.000000,0.000000,0.000000,334.000000,0.000000,0.000000,334.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000,0.000000
25%,364.500000,20.000000,60.000000,7537.500000,5.000000,5.000000,1954.000000,1966.000000,0.00000,0.000000,0.000000,222.000000,795.000000,882.000000,0.000000,0.000000,1128.000000,0.000000,0.000000,1.000000,0.000000,2.000000,1.000000,5.000000,0.000000,1958.000000,1.000000,327.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129900.000000,1.000000
50%,729.000000,50.000000,69.000000,9477.000000,6.000000,5.000000,1972.000000,1993.000000,0.00000,381.000000,0.000000,479.000000,991.000000,1086.000000,0.000000,0.000000,1464.000000,0.000000,0.000000,2.000000,0.000000,3.000000,1.000000,6.000000,1.000000,1977.000000,2.000000,478.000000,0.000000,24.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,162500.000000,1.000000
75%,1094.500000,70.000000,79.000000,11600.000000,7.000000,6.000000,2000.000000,2004.000000,166.00000,707.000000,0.000000,808.000000,1297.500000,1391.000000,728.000000,0.000000,1776.000000,1.000000,0.000000,2.000000,1.000000,3.000000,1.000000,7.000000,1.000000,2001.000000,2.000000,576.000000,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000,1.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.00000,5644.000000,1474.000000,2336.000000,6110.000000,4692.000000,2065.000000,572.000000,5642.000000,3.000000,2.000000,3.000000,2.000000,8.000000,3.000000,14.000000,3.000000,2010.000000,4.000000,1418.000000,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000,1.000000


应该特殊对待，或者不适合作为数字运算的，经过排查，我认为只有：
* Id
* MSSubClass

然后是MoSold，这个是销售月份，我认为后续可以跟YrSold合成一个变量，单独的MoSold意义有限。


我们这里汇总一下处理方法。首先是数值变量

* Id应该直接删掉，在模型训练中没有意义。但是注意提交的时候还需要，所以不能不管。
* MSSubClass应该变成字符串变量，甚至重命名

然后是类别变量。我现在想了一下，干脆我们直接把那些具有明确意义的类别变量，全都换成数值吧。也就是说，我们接受等距假设，虽然严格讲这是Feature Engineering的部分，但是我们姑且就这么定了。还有一些变量总归是需要encoding的。

首先，我们处理两个数值变量

In [15]:
# 删除Id，保存
df_train["Id"].to_csv(PROCESSED_DIR / "Id.csv", index=False)
df_train.drop("Id", axis=1, inplace=True)

# 替换MSSubClass的类型
df_train["MSSubClass"] = df_train["MSSubClass"].astype("str")

print(df_train.shape)
print(df_train.dtypes)

(1451, 81)
MSSubClass           str
MSZoning             str
LotFrontage      float64
LotArea            int64
Street               str
Alley                str
LotShape             str
LandContour          str
Utilities            str
LotConfig            str
LandSlope            str
Neighborhood         str
Condition1           str
Condition2           str
BldgType             str
HouseStyle           str
OverallQual        int64
OverallCond        int64
YearBuilt          int64
YearRemodAdd       int64
RoofStyle            str
RoofMatl             str
Exterior1st          str
Exterior2nd          str
MasVnrType           str
MasVnrArea       float64
ExterQual            str
ExterCond            str
Foundation           str
BsmtQual             str
BsmtCond             str
BsmtExposure         str
BsmtFinType1         str
BsmtFinSF1         int64
BsmtFinType2         str
BsmtFinSF2         int64
BsmtUnfSF          int64
TotalBsmtSF        int64
Heating              str
HeatingQC     

接下来，就需要处理那些有顺序的类别变量，我们把他们统统换成数字

In [16]:
# * LotShape
df_train["LotShape"] = df_train["LotShape"].replace(
    {
        "Reg": 1,
        "IR1": 2,
        "IR2": 3,
        "IR3": 4
    }
)
df_train["LotShape"] = df_train["LotShape"].astype(int)

# * Utilities
df_train["Utilities"] = df_train["Utilities"].replace(
    {
        "AllPub": 4,
        "NoSewr": 3,
        "NoSeWa": 2,
        "ELO": 1
    }
)
df_train["Utilities"] = df_train["Utilities"].astype(int)

# * LandSlope
df_train["LandSlope"] = df_train["LandSlope"].replace(
    {
        "Gtl": 1,
        "Mod": 2,
        "Sev": 3
    }
)
df_train["LandSlope"] = df_train["LandSlope"].astype(int)

# * ExterQual
df_train["ExterQual"] = df_train["ExterQual"].replace(
    {
        "Ex": 5,
        "Gd": 4,
        "TA": 3,
        "Fa": 2,
        "Po": 1
    }
)
df_train["ExterQual"] = df_train["ExterQual"].astype(int)

# * ExterCond
df_train["ExterCond"] = df_train["ExterCond"].replace(
    {
        "Ex": 5,
        "Gd": 4,
        "TA": 3,
        "Fa": 2,
        "Po": 1
    }
)
df_train["ExterCond"] = df_train["ExterCond"].astype(int)

# * BsmtQual
df_train["BsmtQual"] = df_train["BsmtQual"].replace(
    {
        "Ex": 5,
        "Gd": 4,
        "TA": 3,
        "Fa": 2,
        "Po": 1,
        "NoBsmt": 0
    }
)
df_train["BsmtQual"] = df_train["BsmtQual"].astype(int)

# * BsmtCond
df_train["BsmtCond"] = df_train["BsmtCond"].replace(
    {
        "Ex": 5,
        "Gd": 4,
        "TA": 3,
        "Fa": 2,
        "Po": 1,
        "NoBsmt": 0
    }
)
df_train["BsmtCond"] = df_train["BsmtCond"].astype(int)

# * BsmtExposure
df_train["BsmtExposure"] = df_train["BsmtExposure"].replace(
    {
        "Gd": 4,
        "Av": 3,
        "Mn": 2,
        "No": 1,
        "NoBsmt": 0
    }
)
df_train["BsmtExposure"] = df_train["BsmtExposure"].astype(int)

# * BsmtFinType1
df_train["BsmtFinType1"] = df_train["BsmtFinType1"].replace(
    {
        "GLQ": 6,
        "ALQ": 5,
        "BLQ": 4,
        "Rec": 3,
        "LwQ": 2,
        "Unf": 1,
        "NoBsmt": 0
    }
)
df_train["BsmtFinType1"] = df_train["BsmtFinType1"].astype(int)

# * BsmtFinType2
df_train["BsmtFinType2"] = df_train["BsmtFinType2"].replace(
    {
        "GLQ": 6,
        "ALQ": 5,
        "BLQ": 4,
        "Rec": 3,
        "LwQ": 2,
        "Unf": 1,
        "NoBsmt": 0
    }
)
df_train["BsmtFinType2"] = df_train["BsmtFinType2"].astype(int)

# * HeatingQC
df_train["HeatingQC"] = df_train["HeatingQC"].replace(
    {
        "Ex": 5,
        "Gd": 4,
        "TA": 3,
        "Fa": 2,
        "Po": 1,
    }
)
df_train["HeatingQC"] = df_train["HeatingQC"].astype(int)

# * KitchenQual
df_train["KitchenQual"] = df_train["KitchenQual"].replace(
    {
        "Ex": 5,
        "Gd": 4,
        "TA": 3,
        "Fa": 2,
        "Po": 1,
    }
)
df_train["KitchenQual"] = df_train["KitchenQual"].astype(int)

# * Functional
df_train["Functional"] = df_train["Functional"].replace(
    {
        "Typ":7,
        "Min1":6,
        "Min2":5,
        "Mod":4,
        "Maj1":3,
        "Maj2":2,
        "Sev":1,
        "Sal":0
    }
)
df_train["Functional"] = df_train["Functional"].astype(int)

# * FireplaceQu
df_train["FireplaceQu"] = df_train["FireplaceQu"].replace(
    {
        "Ex": 5,
        "Gd": 4,
        "TA": 3,
        "Fa": 2,
        "Po": 1,
        "NoFireplace": 0
    }
)
df_train["FireplaceQu"] = df_train["FireplaceQu"].astype(int)

# * GarageFinish
df_train["GarageFinish"] = df_train["GarageFinish"].replace(
    {
        "Fin": 3,
        "RFn": 2,
        "Unf": 1,
        "NoGarage": 0
    }
)
df_train["GarageFinish"] = df_train["GarageFinish"].astype(int)

# * GarageQual
df_train["GarageQual"] = df_train["GarageQual"].replace(
    {
        "Ex": 5,
        "Gd": 4,
        "TA": 3,
        "Fa": 2,
        "Po": 1,
        "NoGarage": 0
    }
)
df_train["GarageQual"] = df_train["GarageQual"].astype(int)

# * GarageCond
df_train["GarageCond"] = df_train["GarageCond"].replace(
    {
        "Ex": 5,
        "Gd": 4,
        "TA": 3,
        "Fa": 2,
        "Po": 1,
        "NoGarage": 0
    }
)
df_train["GarageCond"] = df_train["GarageCond"].astype(int)

# * PavedDrive
df_train["PavedDrive"] = df_train["PavedDrive"].replace(
    {
        "Y": 2,
        "P": 1,
        "N": 0
    }
)
df_train["PavedDrive"] = df_train["PavedDrive"].astype(int)

# * PoolQC
df_train["PoolQC"] = df_train["PoolQC"].replace(
    {
        "Ex": 4,
        "Gd": 3,
        "TA": 2,
        "Fa": 1,
        "NoPool":0
    }
)
df_train["PoolQC"] = df_train["PoolQC"].astype(int)
# * Fence
df_train["Fence"] = df_train["Fence"].replace(
    {
        "GdPrv": 4,
        "MnPrv": 3,
        "GdWo": 2,
        "MnWw": 1,
        "NoFence":0
    }
)
df_train["Fence"] = df_train["Fence"].astype(int)

现在检查一下替换结果

In [17]:
pd.reset_option("display.max_rows")
df_train.select_dtypes("str")

,MSSubClass,MSZoning,Street,Alley,LandContour,LotConfig,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,Foundation,Heating,CentralAir,Electrical,GarageType,MiscFeature,SaleType,SaleCondition
0,60,RL,Pave,NoAlley,Lvl,Inside,CollgCr,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,BrkFace,PConc,GasA,Y,SBrkr,Attchd,NoMisc,WD,Normal
1,20,RL,Pave,NoAlley,Lvl,FR2,Veenker,Feedr,Norm,1Fam,1Story,Gable,CompShg,MetalSd,MetalSd,NoMasVnr,CBlock,GasA,Y,SBrkr,Attchd,NoMisc,WD,Normal
2,60,RL,Pave,NoAlley,Lvl,Inside,CollgCr,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,BrkFace,PConc,GasA,Y,SBrkr,Attchd,NoMisc,WD,Normal
3,70,RL,Pave,NoAlley,Lvl,Corner,Crawfor,Norm,Norm,1Fam,2Story,Gable,CompShg,Wd Sdng,Wd Shng,NoMasVnr,BrkTil,GasA,Y,SBrkr,Detchd,NoMisc,WD,Abnorml
4,60,RL,Pave,NoAlley,Lvl,FR2,NoRidge,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,BrkFace,PConc,GasA,Y,SBrkr,Attchd,NoMisc,WD,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,60,RL,Pave,NoAlley,Lvl,Inside,Gilbert,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,NoMasVnr,PConc,GasA,Y,SBrkr,Attchd,NoMisc,WD,Normal
1456,20,RL,Pave,NoAlley,Lvl,Inside,NWAmes,Norm,Norm,1Fam,1Story,Gable,CompShg,Plywood,Plywood,Stone,CBlock,GasA,Y,SBrkr,Attchd,NoMisc,WD,Normal
1457,70,RL,Pave,NoAlley,Lvl,Inside,Crawfor,Norm,Norm,1Fam,2Story,Gable,CompShg,CemntBd,CmentBd,NoMasVnr,Stone,GasA,Y,SBrkr,Attchd,Shed,WD,Normal
1458,20,RL,Pave,NoAlley,Lvl,Inside,NAmes,Norm,Norm,1Fam,1Story,Hip,CompShg,MetalSd,MetalSd,NoMasVnr,CBlock,GasA,Y,FuseA,Attchd,NoMisc,WD,Normal


In [18]:
df_train.select_dtypes("number")

,LotFrontage,LotArea,LotShape,Utilities,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,HeatingQC,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,Bedroom,Kitchen,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscVal,MoSold,YrSold,SalePrice,HasGarage
0,65.0,8450,1,4,1,7,5,2003,2003,196.0,4,3,4,3,1,6,706,1,0,150,856,5,856,854,0,1710,1,0,2,1,3,1,4,8,7,0,0,2003,2,2,548,3,3,2,0,61,0,0,0,0,0,0,0,2,2008,208500,1
1,80.0,9600,1,4,1,6,8,1976,1976,0.0,3,3,4,3,4,5,978,1,0,284,1262,5,1262,0,0,1262,0,1,2,0,3,1,3,6,7,1,3,1976,2,2,460,3,3,2,298,0,0,0,0,0,0,0,0,5,2007,181500,1
2,68.0,11250,2,4,1,7,5,2001,2002,162.0,4,3,4,3,2,6,486,1,0,434,920,5,920,866,0,1786,1,0,2,1,3,1,4,6,7,1,3,2001,2,2,608,3,3,2,0,42,0,0,0,0,0,0,0,9,2008,223500,1
3,60.0,9550,2,4,1,7,5,1915,1970,0.0,3,3,3,4,1,5,216,1,0,540,756,4,961,756,0,1717,1,0,1,0,3,1,4,7,7,1,4,1998,1,3,642,3,3,2,0,35,272,0,0,0,0,0,0,2,2006,140000,1
4,84.0,14260,2,4,1,8,5,2000,2000,350.0,4,3,4,3,3,6,655,1,0,490,1145,5,1145,1053,0,2198,1,0,2,1,4,1,4,9,7,1,3,2000,2,3,836,3,3,2,192,84,0,0,0,0,0,0,0,12,2008,250000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,62.0,7917,1,4,1,6,5,1999,2000,0.0,3,3,4,3,1,1,0,1,0,953,953,5,953,694,0,1647,0,0,2,1,3,1,3,7,7,1,3,1999,2,2,460,3,3,2,0,40,0,0,0,0,0,0,0,8,2007,175000,1
1456,85.0,13175,1,4,1,6,6,1978,1988,119.0,3,3,4,3,1,5,790,3,163,589,1542,3,2073,0,0,2073,1,0,2,0,3,1,3,7,6,2,3,1978,1,2,500,3,3,2,349,0,0,0,0,0,0,3,0,2,2010,210000,1
1457,66.0,9042,1,4,1,7,9,1941,2006,0.0,5,4,3,4,1,6,275,1,0,877,1152,5,1188,1152,0,2340,0,0,2,0,4,1,4,9,7,2,4,1941,2,1,252,3,3,2,0,60,0,0,0,0,0,4,2500,5,2010,266500,1
1458,68.0,9717,1,4,1,5,6,1950,1996,0.0,3,3,3,3,2,6,49,3,1029,0,1078,4,1078,0,0,1078,1,0,1,0,2,1,4,5,7,0,0,1950,1,1,240,3,3,2,366,0,112,0,0,0,0,0,0,4,2010,142125,1


到目前为止，预处理阶段就只剩下scaling和encoding了。不过onehot encoding之后不适合做feature engineering了，我们这里先保存一下数据，方便构造feature。至于后面的代码，也不是白写，因为总归到最后是需要固定到.py里面的，代码先跑通，最后固定就行了。

In [19]:
df_train.to_csv(PROCESSED_DIR / "train_cleaned.csv", index=False)

## Scaling and One hot Encoding
到目前为止，我们已经做完初步的encoding了，当然还有明确的类别变量需要进一步使用诸如one hot encoding等处理，但那个我们打算到feature engineering部分再做。现在我们先把scale统一一下，preprocessing就可以收工了。

我打算使用标准化来做scaling，将使用`sklearn.preprocessing`的`RobustScaler`函数。

首先确认一下数据类型

In [20]:
df_train.dtypes.value_counts()

int64      55
str        24
float64     2
Name: count, dtype: int64

接着我们对数值列做变换。在变换前先把SalePrice做一下对数变换。然后再使用RobustScaler变换。这里的变换直接把二元变量，年份，和已经Ordinal encoded过后的分类变量都直接变换了。

In [21]:
import numpy as np
df_train["SalePrice"] = np.log(df_train["SalePrice"])

In [22]:
from sklearn.preprocessing import StandardScaler, RobustScaler
scaler = RobustScaler()

In [23]:
num_columns = df_train.select_dtypes("number").columns
df_train[num_columns] = scaler.fit_transform(df_train[num_columns])

接着，我们需要给那些真正的，不能Ordinal encoding的分类变量编码，在编码前先看一下各自的类别数量

In [24]:
df_train.select_dtypes("str").nunique()

MSSubClass       15
MSZoning          5
Street            2
Alley             3
LandContour       4
LotConfig         5
Neighborhood     25
Condition1        9
Condition2        8
BldgType          5
HouseStyle        8
RoofStyle         6
RoofMatl          8
Exterior1st      15
Exterior2nd      16
MasVnrType        4
Foundation        6
Heating           6
CentralAir        2
Electrical        5
GarageType        7
MiscFeature       5
SaleType          9
SaleCondition     6
dtype: int64

大部分都是十几二十个，感觉还行，没太大问题，开始onehot encoding

In [25]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)

# 首先获取列名
cat_columns = df_train.select_dtypes("str")
cat_columns_name = cat_columns.columns.tolist()
# print(cat_columns_name)

# 编码
encoded = encoder.fit_transform(cat_columns)
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(cat_columns_name),
    index=df_train.index
)

df_train = pd.concat([df_train.drop(columns=cat_columns_name), encoded_df], axis=1)

这样，我们就完成了scaling 和onehot encoding

In [26]:
df_train.describe()

,LotFrontage,LotArea,LotShape,Utilities,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,HeatingQC,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,Bedroom,Kitchen,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscVal,MoSold,YrSold,SalePrice,HasGarage,MSSubClass_120,MSSubClass_160,MSSubClass_180,MSSubClass_190,MSSubClass_20,MSSubClass_30,MSSubClass_40,MSSubClass_45,MSSubClass_50,MSSubClass_60,MSSubClass_70,MSSubClass_75,MSSubClass_80,MSSubClass_85,MSSubClass_90,MSZoning_C (all),MSZoning_FV,MSZoning_RH,MSZoning_RL,MSZoning_RM,Street_Grvl,Street_Pave,Alley_Grvl,Alley_NoAlley,Alley_Pave,LandContour_Bnk,LandContour_HLS,LandContour_Low,LandContour_Lvl,LotConfig_Corner,LotConfig_CulDSac,LotConfig_FR2,LotConfig_FR3,LotConfig_Inside,Neighborhood_Blmngtn,Neighborhood_Blueste,Neighborhood_BrDale,Neighborhood_BrkSide,Neighborhood_ClearCr,Neighborhood_CollgCr,Neighborhood_Crawfor,Neighborhood_Edwards,Neighborhood_Gilbert,Neighborhood_IDOTRR,Neighborhood_MeadowV,Neighborhood_Mitchel,Neighborhood_NAmes,Neighborhood_NPkVill,Neighborhood_NWAmes,Neighborhood_NoRidge,Neighborhood_NridgHt,Neighborhood_OldTown,Neighborhood_SWISU,Neighborhood_Sawyer,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker,Condition1_Artery,Condition1_Feedr,Condition1_Norm,Condition1_PosA,Condition1_PosN,Condition1_RRAe,Condition1_RRAn,Condition1_RRNe,Condition1_RRNn,Condition2_Artery,Condition2_Feedr,Condition2_Norm,Condition2_PosA,Condition2_PosN,Condition2_RRAe,Condition2_RRAn,Condition2_RRNn,BldgType_1Fam,BldgType_2fmCon,BldgType_Duplex,BldgType_Twnhs,BldgType_TwnhsE,HouseStyle_1.5Fin,HouseStyle_1.5Unf,HouseStyle_1Story,HouseStyle_2.5Fin,HouseStyle_2.5Unf,HouseStyle_2Story,HouseStyle_SFoyer,HouseStyle_SLvl,RoofStyle_Flat,RoofStyle_Gable,RoofStyle_Gambrel,RoofStyle_Hip,RoofStyle_Mansard,RoofStyle_Shed,RoofMatl_ClyTile,RoofMatl_CompShg,RoofMatl_Membran,RoofMatl_Metal,RoofMatl_Roll,RoofMatl_Tar&Grv,RoofMatl_WdShake,RoofMatl_WdShngl,Exterior1st_AsbShng,Exterior1st_AsphShn,Exterior1st_BrkComm,Exterior1st_BrkFace,Exterior1st_CBlock,Exterior1st_CemntBd,Exterior1st_HdBoard,Exterior1st_ImStucc,Exterior1st_MetalSd,Exterior1st_Plywood,Exterior1st_Stone,Exterior1st_Stucco,Exterior1st_VinylSd,Exterior1st_Wd Sdng,Exterior1st_WdShing,Exterior2nd_AsbShng,Exterior2nd_AsphShn,Exterior2nd_Brk Cmn,Exterior2nd_BrkFace,Exterior2nd_CBlock,Exterior2nd_CmentBd,Exterior2nd_HdBoard,Exterior2nd_ImStucc,Exterior2nd_MetalSd,Exterior2nd_Other,Exterior2nd_Plywood,Exterior2nd_Stone,Exterior2nd_Stucco,Exterior2nd_VinylSd,Exterior2nd_Wd Sdng,Exterior2nd_Wd Shng,MasVnrType_BrkCmn,MasVnrType_BrkFace,MasVnrType_NoMasVnr,MasVnrType_Stone,Foundation_BrkTil,Foundation_CBlock,Foundation_PConc,Foundation_Slab,Foundation_Stone,Foundation_Wood,Heating_Floor,Heating_GasA,Heating_GasW,Heating_Grav,Heating_OthW,Heating_Wall,CentralAir_N,CentralAir_Y,Electrical_FuseA,Electrical_FuseF,Electrical_FuseP,Electrical_Mix,Electrical_SBrkr,GarageType_2Types,GarageType_Attchd,GarageType_Basment,GarageType_BuiltIn,GarageType_CarPort,GarageType_Detchd,GarageType_NoGarage,MiscFeature_Gar2,MiscFeature_NoMisc,MiscFeature_Othr,MiscFeature_Shed,MiscFeature_TenC,SaleType_COD,SaleType_CWD,SaleType_Con,SaleType_ConLD,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
count,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.00000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000,1451.000000

可见这些列已经处理好了，现在自动拓宽了，我们看一下当前数据表的形状：

In [27]:
df_train.shape

(1451, 241)

现在数据表拓宽到241列，因为onehot encoding会拓宽列，不过这不会影响训练，因为这些列本质上都是一个特征，信息密度没有变。最后，我们把数据保存，就可以进入下一步，feature engineering了

In [28]:
df_train.to_csv(PROCESSED_DIR / "preprocessed_without_FE.csv", index=False)